In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

import joblib

In [5]:
import os
print(os.getcwd())

c:\Users\avins\Downloads\Smart Agri AI\ml_models\soil_moisture


In [6]:
df = pd.read_csv("../../datasets/soil moisture/soil_moisture_dataset.csv")

print("Dataset Preview:")
print(df.head())

Dataset Preview:
   Temperature  Humidity  Rainfall  Sunlight  Crop_Type  Crop_Stage  \
0        32.79     41.25      5.50       Low      Wheat    Seedling   
1        37.84     44.35      8.44       Low       Rice  Vegetative   
2        32.04     68.06     14.32      High     Cotton    Fruiting   
3        31.79     80.47      0.13       Low  Sugarcane   Flowering   
4        24.31     78.17      2.04  Moderate       Rice   Flowering   

   Soil_Moisture  
0          42.07  
1          36.32  
2          46.82  
3          30.44  
4          44.05  


In [7]:
# Separate categorical & numerical
categorical_cols = ["Sunlight", "Crop_Type", "Crop_Stage"]
numerical_cols = ["Temperature", "Humidity", "Rainfall"]

# OneHot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[categorical_cols])

# Convert to DataFrame
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))

# Combine with numerical data
X = pd.concat([df[numerical_cols], encoded_df], axis=1)

y = df["Soil_Moisture"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
rf = RandomForestRegressor(random_state=42)

param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 12, 15],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [2, 3],
    "max_features": ["sqrt"]
}

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

# Best model
model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Parameters: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 150}


In [34]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n📊 Model Performance:")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")



📊 Model Performance:
MAE  : 4.54
RMSE : 5.47
R²   : 0.8786


In [35]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f"Train Score: {train_score:.4f}")
print(f"Test Score : {test_score:.4f}")

Train Score: 0.9532
Test Score : 0.8786


In [36]:
joblib.dump(
    model,
    "soil_moisture_model.pkl",
    compress=3
)
joblib.dump(encoder, "encoders.pkl")

print("\n✅ Model saved successfully!")


✅ Model saved successfully!


In [37]:
def predict_soil_moisture(temp, humidity, rainfall, sunlight, crop, stage):
    
    model = joblib.load("soil_moisture_model.pkl")
    encoder = joblib.load("encoders.pkl")

    input_df = pd.DataFrame([{
        "Temperature": temp,
        "Humidity": humidity,
        "Rainfall": rainfall,
        "Sunlight": sunlight,
        "Crop_Type": crop,
        "Crop_Stage": stage
    }])

    categorical_cols = ["Sunlight", "Crop_Type", "Crop_Stage"]
    numerical_cols = ["Temperature", "Humidity", "Rainfall"]

    encoded = encoder.transform(input_df[categorical_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))

    final_input = pd.concat([input_df[numerical_cols], encoded_df], axis=1)

    prediction = model.predict(final_input)[0]

    return round(prediction, 2)

In [38]:

sample = predict_soil_moisture(
    temp=38,
    humidity=60,
    rainfall=4,
    sunlight="High",
    crop="Wheat",
    stage="Harvesting"
)

print("\n🌱 Sample Prediction:", sample)


🌱 Sample Prediction: 24.77
